# Handwritten music symbol detector: train on Kaggle (free GPU)

1. **Settings → Accelerator: GPU T4 ×1** (or P100). **Settings → Internet: On** (needs a phone-verified Kaggle account).
2. Optional: to add your copyist's pages, attach a Kaggle Dataset containing `copyist_train.json`, `copyist_val.json` and their tile images (made with `prepare_copyist.py`, see the README). Set `COPYIST_DIR` below.
3. Click **Save Version → Save & Run All (Commit)**. That runs in the background, so you can close the tab. The YOLOX-Tiny run takes a few hours (estimate).
4. When it finishes, open the version's **Output** tab and download `export/`: `music_symbols.onnx`, `music_symbols.fp16.onnx` and `music_symbols.json`.

Data license: MUSCIMA++ / CVC-MUSCIMA are CC BY-NC-SA 4.0, so the trained model is for **non-commercial** use and must credit them.

In [ ]:
MODEL = 'tiny'          # 'tiny' (more accurate) or 'nano' (smaller, faster)
EPOCHS = 120
SYNTH_TILES = 8000
COPYIST_DIR = ''        # e.g. '/kaggle/input/my-copyist-pages' (optional)
BASE_CKPT = ''          # e.g. '/kaggle/input/music-base/best_ckpt.pth' to fine-tune a previous run (then ~40 epochs is enough)
BRANCH = 'claude/festive-cray-j8gcf7'

In [ ]:
%cd /kaggle/working
!git clone -q --depth 1 -b {BRANCH} https://github.com/nmd2827/grants-hub repo
!git clone -q --depth 1 https://github.com/Megvii-BaseDetection/YOLOX yolox
!git clone -q --depth 1 https://github.com/steinbergmedia/petaluma petaluma
!pip install -q loguru tabulate thop ninja pycocotools onnx onnxsim onnxconverter-common
!cd yolox && pip install -q --no-build-isolation -e . 2>&1 | tail -2
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - enable it in Settings')

In [ ]:
%cd /kaggle/working/repo/pdf-to-flat/training
!python prepare_muscima.py --download --root /kaggle/working/data/muscima --out /kaggle/working/data/coco
!python synth_petaluma.py --font /kaggle/working/petaluma/redist/otf/Petaluma.otf --out /kaggle/working/data/coco --train {SYNTH_TILES} --val 300
if COPYIST_DIR:
    !cp -r {COPYIST_DIR}/train2017/. /kaggle/working/data/coco/train2017/ ; cp -r {COPYIST_DIR}/val2017/. /kaggle/working/data/coco/val2017/ ; cp {COPYIST_DIR}/*.json /kaggle/working/data/coco/annotations/
!python merge_coco.py --out /kaggle/working/data/coco
!python viz.py /kaggle/working/data/coco/annotations/instances_train2017.json /kaggle/working/data/coco/train2017 /kaggle/working/viz 4

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('/kaggle/working/viz/*.png'))[:4]:
    display(Image(f))

In [ ]:
import os
os.environ['MUSIC_DATA_DIR'] = '/kaggle/working/data/coco'
os.environ['MUSIC_MODEL'] = MODEL
os.environ['MUSIC_EPOCHS'] = str(EPOCHS)
%cd /kaggle/working/yolox
extra = f'-c {BASE_CKPT}' if BASE_CKPT else ''
!python tools/train.py -f /kaggle/working/repo/pdf-to-flat/training/yolox_handwritten.py -d 1 -b 32 --fp16 -o --cache ram {extra}

In [ ]:
# held-out writers (MUSCIMA++ writers 46-50 + copyist test pages if any)
ckpt = f'/kaggle/working/yolox/YOLOX_outputs/music_{MODEL}/best_ckpt.pth'
!python tools/eval.py -f /kaggle/working/repo/pdf-to-flat/training/yolox_handwritten.py -c {ckpt} -b 16 -d 1 --conf 0.001 --test 2>&1 | tail -40

In [ ]:
%cd /kaggle/working/yolox
!python /kaggle/working/repo/pdf-to-flat/training/export_model.py --exp /kaggle/working/repo/pdf-to-flat/training/yolox_handwritten.py --ckpt {ckpt} --out /kaggle/working/export
!python /kaggle/working/repo/pdf-to-flat/training/per_class_report.py --data /kaggle/working/data/coco --split test2017 --onnx /kaggle/working/export/music_symbols.onnx | tee /kaggle/working/export/test_report.txt
!cp {ckpt} /kaggle/working/export/best_ckpt.pth